<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/Control%20Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Test


In [ ]:
# @title Env

!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

# %%
from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)

# %%
# Optional: install control (not used in this notebook but kept for compatibility)
!pip install -q control

# ## 2. Load Model (IDF + Weather)


import types, datetime, requests, io, os, gc
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from eplus.core import EPlusUtil

In [32]:
# @title Setup


OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAirCooled_Exp.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

# ## 3. Register Variables for Logging

# %%
# Define all variables we want to log: real values and reference/setpoints.
# We'll use "*" for key where we want per-zone values, else specify component name.
specs = [
    # --- Zone conditions (real) ---
    {"name": "Zone Mean Air Temperature", "key": "*"},
    {"name": "Zone Mean Radiant Temperature", "key": "*"},
    {"name": "Zone Mean Air Humidity Ratio", "key": "*"},
    {"name": "Zone Air CO2 Concentration", "key": "*"},
    {"name": "Zone People Occupant Count", "key": "*"},

    # --- Supply system real values ---
    {"name": "System Node Temperature", "key": "VAV Sys 1 Outlet Node"},
    {"name": "System Node Humidity Ratio", "key": "VAV Sys 1 Outlet Node"},
    {"name": "System Node CO2 Concentration", "key": "VAV Sys 1 Outlet Node"},
    {"name": "System Node Mass Flow Rate", "key": "VAV Sys 1 Outlet Node"},

    # --- Zone air inlet (after VAV+reheat) real flows ---
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE1-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE2-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE3-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE4-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE5-1 In Node"},

    # --- Hot water coil flows (reheat + main + OA heat) ---
    {"name": "System Node Mass Flow Rate", "key": "SPACE1-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE2-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE3-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE4-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE5-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "Main Heating Coil 1 Water Inlet Node"},
    {"name": "System Node Mass Flow Rate", "key": "OA Heating Coil 1 Water Inlet Node"},

    # --- Chilled water coil flows ---
    {"name": "System Node Mass Flow Rate", "key": "Main Cooling Coil 1 Water Inlet Node"},
    {"name": "System Node Mass Flow Rate", "key": "OA Cooling Coil 1 Water Inlet Node"},

    # --- Plant flows ---
    {"name": "Pump Mass Flow Rate", "key": "CW CIRC PUMP"},
    {"name": "Pump Mass Flow Rate", "key": "HW CIRC PUMP"},

    # --- Chiller & Boiler status ---
    {"name": "Cooling Coil Total Cooling Rate", "key": "Main Cooling Coil 1"},
    {"name": "Boiler Heating Rate", "key": "Central Boiler"},
    {"name": "Chiller Evaporator Cooling Rate", "key": "Central Chiller"},

    # --- Outdoor environment ---
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "*"},
    {"name": "Site Outdoor Air Humidity Ratio", "key": "*"},
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},  # outdoor CO2 ppm

    # --- Reference/setpoint values (we will fill these from our control handler) ---
    # We'll log them manually in the logger.
]

sim.ensure_output_variables(specs, activate=True)


# ## 4. Data Logger

# %%
sim.collected_data = []
sim.current_state = {}

def state_logger(self, state):
    """Collect all real values and also our applied setpoints (stored in self.control_setpoints)."""
    if not self.exchange.api_data_fully_ready(state):
        return

    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    total_minutes = int(time_now * 60)
    hours, mins = divmod(total_minutes, 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # Helper to get variable value
    def get_val(name, key):
        handle = self.exchange.get_variable_handle(state, name, key)
        return self.exchange.get_variable_value(state, handle) if handle != -1 else np.nan

    # --- Outdoor ---
    row["T_out"] = get_val("Site Outdoor Air Drybulb Temperature", "Environment")
    row["W_out"] = get_val("Site Outdoor Air Humidity Ratio", "Environment")
    row["CO2_out"] = get_val("Schedule Value", "CO2-Outdoor-Actuated")

    # --- Supply ---
    row["T_supply"] = get_val("System Node Temperature", "VAV Sys 1 Outlet Node")
    row["W_supply"] = get_val("System Node Humidity Ratio", "VAV Sys 1 Outlet Node")
    row["CO2_supply"] = get_val("System Node CO2 Concentration", "VAV Sys 1 Outlet Node")
    row["M_supply"] = get_val("System Node Mass Flow Rate", "VAV Sys 1 Outlet Node")

    # --- Zone conditions & flows ---
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
    for z in zones:
        row[f"{z}_T_in"] = get_val("Zone Mean Air Temperature", z)
        row[f"{z}_T_m"]   = get_val("Zone Mean Radiant Temperature", z)
        row[f"{z}_W_in"]  = get_val("Zone Mean Air Humidity Ratio", z)
        row[f"{z}_CO2"]   = get_val("Zone Air CO2 Concentration", z)
        row[f"{z}_Occ"]   = get_val("Zone People Occupant Count", z)
        row[f"{z}_V_dot"] = get_val("System Node Current Density Volume Flow Rate", f"{z} In Node")

    # --- Reheat coil water flows ---
    for z in zones:
        row[f"{z}_reheat_water_mdot"] = get_val("System Node Mass Flow Rate", f"{z} Zone Coil Water In Node")

    # --- Main heating coil water flow ---
    row["main_heat_water_mdot"] = get_val("System Node Mass Flow Rate", "Main Heating Coil 1 Water Inlet Node")

    # --- OA heating coil water flow ---
    row["oa_heat_water_mdot"] = get_val("System Node Mass Flow Rate", "OA Heating Coil 1 Water Inlet Node")

    # --- Main cooling coil water flow ---
    row["main_cool_water_mdot"] = get_val("System Node Mass Flow Rate", "Main Cooling Coil 1 Water Inlet Node")

    # --- OA cooling coil water flow ---
    row["oa_cool_water_mdot"] = get_val("System Node Mass Flow Rate", "OA Cooling Coil 1 Water Inlet Node")

    # --- Pump flows ---
    row["cw_pump_mdot"] = get_val("Pump Mass Flow Rate", "CW CIRC PUMP")
    row["hw_pump_mdot"] = get_val("Pump Mass Flow Rate", "HW CIRC PUMP")

    # --- Coil loads ---
    row["main_cool_coil_rate"] = get_val("Cooling Coil Total Cooling Rate", "Main Cooling Coil 1")
    row["chiller_evap_rate"] = get_val("Chiller Evaporator Cooling Rate", "Central Chiller")
    row["boiler_heat_rate"] = get_val("Boiler Heating Rate", "Central Boiler")

    # --- Append our applied setpoints (stored by god_mode_control) ---
    sp = getattr(self, 'control_setpoints', {})
    row.update(sp)   # add all keys from setpoint dict

    self.collected_data.append(row)

sim.state_logger = types.MethodType(state_logger, sim)


# ## 5. Occupancy Injector (from CSV)

# %%
def preload_occupancy_csv(sim_obj, url):
    print(f"Downloading CSV from: {url}...")
    resp = requests.get(url)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    if 'timestamp' not in df.columns:
        raise ValueError("CSV must contain a 'timestamp' column.")
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')
    midnight_start = df['timestamp'].iloc[0].normalize()
    df['rel_seconds'] = (df['timestamp'] - midnight_start).dt.total_seconds()
    sim_obj._occ_duration_sec = 86400.0
    df = df.set_index('rel_seconds')
    sim_obj._preloaded_occ_df = df.drop(columns=['timestamp'])
    zones = list(sim_obj._preloaded_occ_df.columns)
    print(f"Success! Preloaded {len(df)} rows. Loop locked to 24.00 hours.")
    print(f"Detected Source Columns: {zones}")

csv_url = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/Occupancy_Dataset.csv"
preload_occupancy_csv(sim, csv_url)

def people_injector(self, state):
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return
    if not hasattr(self, '_fast_injector_ready'):
        if not hasattr(self, '_preloaded_occ_df'):
            print("[Injector] ERROR: Data not preloaded.")
            self._fast_injector_ready = False
            return
        self._zone_occ_rules = {
            "SPACE1-1": {"source": "SPACE1-1", "mult": 1.0, "min": 0, "max": 5},
            "SPACE2-1": {"source": "SPACE1-1", "mult": 1.5, "min": 0, "max": 4},
            "SPACE3-1": {"source": "SPACE1-1", "mult": 0.4, "min": 0, "max": 1},
            "SPACE4-1": {"source": "SPACE1-1", "mult": 1.2, "min": 0, "max": 3},
            "SPACE5-1": {"source": "SPACE1-1", "mult": 2.0, "min": 0, "max": 6},
        }
        self._people_handles = {}
        target_zones = list(self._zone_occ_rules.keys())
        try:
            ep_people_names = self.exchange.get_object_names(state, "People") or []
        except Exception:
            ep_people_names = []
        for z in target_zones:
            matched = [p for p in ep_people_names if z.replace(" ","").lower() in p.replace(" ","").lower()]
            handles = []
            for p in matched:
                h = self.exchange.get_actuator_handle(state, "People", "Number of People", p)
                if h != -1:
                    handles.append(h)
            if handles:
                self._people_handles[z] = handles
        day = self.exchange.day_of_year(state)
        time_hr = self.exchange.current_time(state)
        self._sim_start_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day-1, seconds=int(time_hr*3600))
        self._fast_injector_ready = True

    if not self._fast_injector_ready or getattr(self, '_occ_duration_sec',0)==0:
        return

    day = self.exchange.day_of_year(state)
    time_hr = self.exchange.current_time(state)
    current_date = datetime.datetime(2002,1,1) + datetime.timedelta(days=day-1, seconds=int(time_hr*3600))
    elapsed_seconds = (current_date - self._sim_start_date).total_seconds()
    loop_sec = elapsed_seconds % self._occ_duration_sec
    df = self._preloaded_occ_df
    valid_indices = df.index[df.index <= loop_sec]
    target_idx = df.index[0] if len(valid_indices)==0 else valid_indices[-1]
    row = df.loc[target_idx]

    for z, handles in self._people_handles.items():
        rule = self._zone_occ_rules.get(z)
        if not rule:
            continue
        src_col = rule["source"]
        if src_col in row:
            base_val = float(row[src_col])
            if base_val == 0:
                val = 0.0
            else:
                calculated = np.ceil(base_val * rule["mult"])
                val = float(np.clip(calculated, rule["min"], rule["max"]))
            per_actuator = val / len(handles)
            for h in handles:
                self.exchange.set_actuator_value(state, h, per_actuator)

sim.people_injector = types.MethodType(people_injector, sim)


# ## 6. Outdoor CO₂ Injection

# %%
def co2_set_outdoor_ppm(self, state, value_ppm=420.0, log_every_minutes=60):
    """Keep outdoor CO₂ at a constant value using the schedule actuator"""
    if not hasattr(self, '_co2_out_handle'):
        self._co2_out_handle = self.exchange.get_actuator_handle(state,
            "Schedule:Compact", "Schedule Value", "CO2-Outdoor-Actuated")
        self._co2_log_interval = log_every_minutes
        self._co2_last_log = -999
    if self._co2_out_handle != -1:
        self.exchange.set_actuator_value(state, self._co2_out_handle, value_ppm)
    # Optional log
    if getattr(self, '_co2_last_log', -999) + self._co2_log_interval <= self.exchange.current_time(state)*60:
        self._co2_last_log = self.exchange.current_time(state)*60

sim.co2_set_outdoor_ppm = types.MethodType(co2_set_outdoor_ppm, sim)

Initialized StateMixin
Initialized EnergyPlus State.
Initialized IDFMixin
Initialized LoggingMixin
Initialized SimulationMixin
Initialized UtilsMixin
Initialized HandlersMixin
Initialized SQLMixin
Initialized ControlMixin
Initialized OccupancyMixin
Initialized ZoneObserverMixin
EnergyPlus state has been reset.
Deleted output directory: /simulation/eplus_out
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/5ZoneAirCooled_Exp.idf', EPW='/simulation/eplus_out/LKA_Colombo-Katunayake.434500_SWERA.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.
Success! Preloaded 10129 rows. Loop locked to 24.00 hours.
Detected Source Columns: ['SPACE1-1']


In [33]:
# @title
def god_mode_control(self, state):
    """Full manual HVAC control with safety overrides."""
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # ================== USER SETPOINTS (non‑zero for testing) ==================
    TARGET_ZONE_FLOW_FRACT = 0.5       # fraction of max zone air flow (0–1)  ← try e.g. 0.5
    TARGET_FAN_FLOW = 0.3              # kg/s (supply fan)

    TARGET_MAIN_CC_WATER = 1.5         # kg/s (main cooling coil)
    TARGET_MAIN_HC_WATER = 0.01        # kg/s (main heating coil)
    TARGET_OA_CC_WATER = 0.2           # kg/s (OA cooling coil)
    TARGET_OA_HC_WATER = 0.01          # kg/s (OA heating coil)
    TARGET_REHEAT_WATER = 0.01         # kg/s (each zone reheat coil)

    TARGET_CW_PUMP_FLOW = 3.0          # kg/s (chilled water pump)
    TARGET_HW_PUMP_FLOW = 1.0          # kg/s (hot water pump)

    CHILLER_ON = 1.0
    BOILER_ON = 1.0

    # ================== SAFETY OVERRIDES ==================
    # Read actual cooling / heating rates to avoid plant damage.
    cooling_rate = 0.0
    heating_rate = 0.0

    h_cool_rate = self.exchange.get_variable_handle(
        state, "Cooling Coil Total Cooling Rate", "Main Cooling Coil 1"
    )
    if h_cool_rate != -1:
        cooling_rate = self.exchange.get_variable_value(state, h_cool_rate)

    h_heat_rate = self.exchange.get_variable_handle(
        state, "Boiler Heating Rate", "Central Boiler"
    )
    if h_heat_rate != -1:
        heating_rate = self.exchange.get_variable_value(state, h_heat_rate)

    # If the cooling coil is not actually cooling, turn off chiller & CW pump
    if cooling_rate < 100.0:          # less than 100 W → no load
        CHILLER_ON = 0.0
        TARGET_CW_PUMP_FLOW = 0.0

    # If the boiler is not providing any heat, turn off boiler & HW pump
    if heating_rate < 100.0:
        BOILER_ON = 0.0
        TARGET_HW_PUMP_FLOW = 0.0

    # ================== 1. VAV DAMPERS (air mass flow fraction) ==================
    for zone_id in ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]:
        sched_name = f"{zone_id} MIN FRACT"
        h = self.exchange.get_actuator_handle(
            state, "Schedule:Compact", "Schedule Value", sched_name
        )
        if h != -1:
            self.exchange.set_actuator_value(state, h, TARGET_ZONE_FLOW_FRACT)

    # ================== 2. SUPPLY FAN ==================
    h_fan = self.exchange.get_actuator_handle(
        state, "Fan", "Fan Air Mass Flow Rate", "SUPPLY FAN 1"
    )
    if h_fan != -1:
        self.exchange.set_actuator_value(state, h_fan, TARGET_FAN_FLOW)

    # Also set the system outlet mass flow setpoint for consistency
    h_sys_out = self.exchange.get_actuator_handle(
        state, "System Node Setpoint", "Air Mass Flow Rate", "VAV SYS 1 OUTLET NODE"
    )
    if h_sys_out != -1:
        self.exchange.set_actuator_value(state, h_sys_out, TARGET_FAN_FLOW)

    # ================== 3. MAIN COOLING COIL ==================
    h_mc_on = self.exchange.get_actuator_handle(
        state, "Plant Component Coil:Cooling:Water",
        "On/Off Supervisory", "MAIN COOLING COIL 1"
    )
    if h_mc_on != -1:
        self.exchange.set_actuator_value(state, h_mc_on, 1.0 if CHILLER_ON else 0.0)
    for c_type in ["Mass Flow Rate Setpoint",
                   "Mass Flow Rate Maximum Available Setpoint",
                   "Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(
            state, "System Node Setpoint", c_type,
            "MAIN COOLING COIL 1 WATER INLET NODE"
        )
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_MAIN_CC_WATER)

    # ================== 4. MAIN HEATING COIL ==================
    h_mh_on = self.exchange.get_actuator_handle(
        state, "Plant Component Coil:Heating:Water",
        "On/Off Supervisory", "MAIN HEATING COIL 1"
    )
    if h_mh_on != -1:
        self.exchange.set_actuator_value(state, h_mh_on, 1.0 if BOILER_ON else 0.0)
    for c_type in ["Mass Flow Rate Setpoint",
                   "Mass Flow Rate Maximum Available Setpoint",
                   "Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(
            state, "System Node Setpoint", c_type,
            "MAIN HEATING COIL 1 WATER INLET NODE"
        )
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_MAIN_HC_WATER)

    # ================== 5. OA COOLING COIL ==================
    h_oc_on = self.exchange.get_actuator_handle(
        state, "Plant Component Coil:Cooling:Water",
        "On/Off Supervisory", "OA COOLING COIL 1"
    )
    if h_oc_on != -1:
        self.exchange.set_actuator_value(state, h_oc_on, 1.0 if CHILLER_ON else 0.0)
    for c_type in ["Mass Flow Rate Setpoint",
                   "Mass Flow Rate Maximum Available Setpoint",
                   "Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(
            state, "System Node Setpoint", c_type,
            "OA COOLING COIL 1 WATER INLET NODE"
        )
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_OA_CC_WATER)

    # ================== 6. OA HEATING COIL ==================
    h_oh_on = self.exchange.get_actuator_handle(
        state, "Plant Component Coil:Heating:Water",
        "On/Off Supervisory", "OA HEATING COIL 1"
    )
    if h_oh_on != -1:
        self.exchange.set_actuator_value(state, h_oh_on, 1.0 if BOILER_ON else 0.0)
    for c_type in ["Mass Flow Rate Setpoint",
                   "Mass Flow Rate Maximum Available Setpoint",
                   "Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(
            state, "System Node Setpoint", c_type,
            "OA HEATING COIL 1 WATER INLET NODE"
        )
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_OA_HC_WATER)

    # ================== 7. REHEAT COILS (zone level) ==================
    for zone_id in ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]:
        node_key = f"{zone_id} ZONE COIL WATER IN NODE"
        for c_type in ["Mass Flow Rate Setpoint",
                       "Mass Flow Rate Maximum Available Setpoint",
                       "Mass Flow Rate Minimum Available Setpoint"]:
            h = self.exchange.get_actuator_handle(
                state, "System Node Setpoint", c_type, node_key
            )
            if h != -1:
                self.exchange.set_actuator_value(state, h, TARGET_REHEAT_WATER)

    # ================== 8. PUMPS ==================
    h_cw_pump = self.exchange.get_actuator_handle(
        state, "Pump", "Pump Mass Flow Rate", "CW CIRC PUMP"
    )
    if h_cw_pump != -1:
        self.exchange.set_actuator_value(state, h_cw_pump, TARGET_CW_PUMP_FLOW)

    h_hw_pump = self.exchange.get_actuator_handle(
        state, "Pump", "Pump Mass Flow Rate", "HW CIRC PUMP"
    )
    if h_hw_pump != -1:
        self.exchange.set_actuator_value(state, h_hw_pump, TARGET_HW_PUMP_FLOW)

    # ================== 9. CHILLER / BOILER ON/OFF ==================
    h_chiller = self.exchange.get_actuator_handle(
        state, "Plant Component Chiller:Electric",
        "On/Off Supervisory", "CENTRAL CHILLER"
    )
    if h_chiller != -1:
        self.exchange.set_actuator_value(state, h_chiller, CHILLER_ON)

    h_boiler = self.exchange.get_actuator_handle(
        state, "Plant Component Boiler:HotWater",
        "On/Off Supervisory", "CENTRAL BOILER"
    )
    if h_boiler != -1:
        self.exchange.set_actuator_value(state, h_boiler, BOILER_ON)

    # ================== LOG SETPOINTS ==================
    self.control_setpoints = {
        "set_TARGET_ZONE_FLOW_FRACT": TARGET_ZONE_FLOW_FRACT,
        "set_TARGET_FAN_FLOW": TARGET_FAN_FLOW,
        "set_MAIN_CC_WATER": TARGET_MAIN_CC_WATER,
        "set_MAIN_HC_WATER": TARGET_MAIN_HC_WATER,
        "set_OA_CC_WATER": TARGET_OA_CC_WATER,
        "set_OA_HC_WATER": TARGET_OA_HC_WATER,
        "set_TARGET_REHEAT_WATER_FLOW": TARGET_REHEAT_WATER,
        "set_CW_PUMP_FLOW": TARGET_CW_PUMP_FLOW,
        "set_HW_PUMP_FLOW": TARGET_HW_PUMP_FLOW,
        "set_CHILLER_ON": CHILLER_ON,
        "set_BOILER_ON": BOILER_ON,
    }

# Attach the method to the simulator instance
sim.god_mode_control = types.MethodType(god_mode_control, sim)

In [34]:
# @title Sim
# ## 8. Register All Handlers

# %%
sim.register_handlers("begin", [
    {"method_name": "state_logger"},
    {"method_name": "co2_set_outdoor_ppm", "kwargs": {"value_ppm": 420.0, "log_every_minutes": 60}},
    {"method_name": "people_injector"},
])

# Register the god-mode controller on the inside_iter hook (runs during HVAC iteration)
sim.register_handlers("inside_iter", [
    {"method_name": "god_mode_control"},
])


# ## 9. Run Simulation (Short Period for Testing)
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 2),           # run for 1 day
    timestep_per_hour=4,  # 15-minute steps
    start_day_of_week="Sunday",
)

print("Starting controlled simulation...")
res = sim.run_annual()

if res == 0:
    print("Simulation complete. Converting data...")
    df = pd.DataFrame(sim.collected_data)
    # Create a proper datetime index for plotting
    sim_start = pd.Timestamp("2026-01-01 00:00:00")
    df['datetime'] = sim_start + pd.to_timedelta(df['day']-1, unit='D') + pd.to_timedelta(df['hour'] + df['minute']/60, unit='h')
    df.set_index('datetime', inplace=True)
    print("DataFrame ready.")
else:
    print("Simulation failed. Check eplusout.err")
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        with open(err_path, 'r') as f:
            print(f.read()[-4000:])


EnergyPlus state has been reset.
Starting controlled simulation...
EnergyPlus state has been reset.
Simulation complete. Converting data...
DataFrame ready.


In [38]:
# @title Plot
def plot_dark_mode_results(df):
    """
    Creates 7 separate dark‑mode plots:
      1. Outdoor, Zone & Supply Temperature
      2. Outdoor, Zone & Supply Humidity Ratio
      3. Outdoor, Zone & Supply CO₂
      4. Zone Air Flows + Total Fan Flow + setpoint refs
      5. Zone Reheat Water Flows + setpoint ref
      6. CHILLER_ON / BOILER_ON setpoints
      7. Coil Water & Pump Flows + their setpoint refs
    """
    dark_template = 'plotly_dark'
    common_layout = dict(template=dark_template, hovermode='x unified')

    zones = [f"SPACE{i}-1" for i in range(1,6)]

    # ---------- Helper to get setpoint values ----------
    def get_setpoint(col_name, default=None):
        if col_name in df.columns and not df[col_name].isna().all():
            return df[col_name].iloc[0]
        return default

    # ---------- Graph 1: Temperatures ----------
    fig1 = go.Figure(layout=common_layout)
    fig1.add_trace(go.Scatter(x=df.index, y=df['T_out'], name='Outdoor T'))
    for z in zones:
        fig1.add_trace(go.Scatter(x=df.index, y=df[f'{z}_T_in'], name=f'{z} T'))
    fig1.add_trace(go.Scatter(x=df.index, y=df['T_supply'], name='Supply T'))
    fig1.update_layout(title="Temperatures", yaxis_title="°C")
    fig1.show()

    # ---------- Graph 2: Humidity Ratios ----------
    fig2 = go.Figure(layout=common_layout)
    fig2.add_trace(go.Scatter(x=df.index, y=df['W_out'], name='Outdoor W'))
    for z in zones:
        fig2.add_trace(go.Scatter(x=df.index, y=df[f'{z}_W_in'], name=f'{z} W'))
    fig2.add_trace(go.Scatter(x=df.index, y=df['W_supply'], name='Supply W'))
    fig2.update_layout(title="Humidity Ratios", yaxis_title="kg/kg")
    fig2.show()

    # ---------- Graph 3: CO₂ ----------
    fig3 = go.Figure(layout=common_layout)
    if 'CO2_out' in df.columns:
        fig3.add_trace(go.Scatter(x=df.index, y=df['CO2_out'], name='Outdoor CO₂'))
    for z in zones:
        col = f'{z}_CO2'
        if col in df.columns:
            fig3.add_trace(go.Scatter(x=df.index, y=df[col], name=f'{z} CO₂'))
    if 'CO2_supply' in df.columns:
        fig3.add_trace(go.Scatter(x=df.index, y=df['CO2_supply'], name='Supply CO₂'))
    fig3.update_layout(title="CO₂ Concentrations", yaxis_title="ppm")
    fig3.show()

    # ---------- Graph 4: Air Flows (zone + fan) ----------
    fig4 = go.Figure(layout=common_layout)
    for z in zones:
        fig4.add_trace(go.Scatter(x=df.index, y=df[f'{z}_V_dot'], name=f'{z} V_dot'))
    fig4.add_trace(go.Scatter(x=df.index, y=df['M_supply'], name='Fan total mass flow'))
    # Reference lines
    zone_flow_set = get_setpoint('set_TARGET_ZONE_FLOW', 0.05)
    fan_flow_set   = get_setpoint('set_TARGET_FAN_FLOW', 0.3)
    fig4.add_hline(y=zone_flow_set, line_dash="dash", line_color="cyan",
                   annotation_text=f"Zone set {zone_flow_set:.2f}")
    fig4.add_hline(y=fan_flow_set, line_dash="dash", line_color="magenta",
                   annotation_text=f"Fan set {fan_flow_set:.2f}")
    fig4.update_layout(title="Air Flow Rates", yaxis_title="m³/s (zone) / kg/s (fan)")
    fig4.show()

    # ---------- Graph 5: Reheat Water Flows ----------
    fig5 = go.Figure(layout=common_layout)
    for z in zones:
        col = f'{z}_reheat_water_mdot'
        if col in df.columns:
            fig5.add_trace(go.Scatter(x=df.index, y=df[col], name=f'{z} reheat'))
    reheat_set = get_setpoint('set_TARGET_REHEAT_WATER_FLOW', 0.0)
    fig5.add_hline(y=reheat_set, line_dash="dash", line_color="cyan",
                   annotation_text=f"Set {reheat_set:.4f}")
    fig5.update_layout(title="Reheat Coil Water Flow Rates", yaxis_title="kg/s")
    fig5.show()

    # ---------- Graph 6: Chiller / Boiler ON setpoints ----------
    fig6 = go.Figure(layout=common_layout)
    # These are always constant 1.0 (we just show the reference)
    chiller_set = get_setpoint('set_CHILLER_ON', 1.0)
    boiler_set  = get_setpoint('set_BOILER_ON', 1.0)
    fig6.add_hline(y=chiller_set, line_dash="dash", line_color="cyan",
                   annotation_text=f"Chiller ON={chiller_set}")
    fig6.add_hline(y=boiler_set, line_dash="dash", line_color="orange",
                   annotation_text=f"Boiler ON={boiler_set}")
    # Optionally show actual chiller/boiler activity if needed (e.g., cooling rate > 0)
    # You could add: fig6.add_trace(go.Scatter(x=df.index, y=(df['chiller_evap_rate']>1).astype(int), ...))
    fig6.update_layout(title="CHILLER / BOILER ON Setpoints", yaxis_title="State", yaxis_range=[0,1.5])
    fig6.show()

    # ---------- Graph 7: Coil & Pump Water Flows (actuals + targets) ----------
    fig7 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                         subplot_titles=["Coil Water Flows", "Pump Flows"],
                         specs=[[{"secondary_y": False}], [{"secondary_y": False}]])
    # Row 1: coil water flows
    coil_flows = ['main_cool_water_mdot', 'main_heat_water_mdot',
                  'oa_cool_water_mdot', 'oa_heat_water_mdot']
    colors = ['cyan', 'orange', 'lime', 'pink']
    for i, col in enumerate(coil_flows):
        if col in df.columns:
            fig7.add_trace(go.Scatter(x=df.index, y=df[col], name=col,
                                      line=dict(color=colors[i])), row=1, col=1)
    # Row 2: pump flows
    pump_flows = ['cw_pump_mdot', 'hw_pump_mdot']
    for col in pump_flows:
        if col in df.columns:
            fig7.add_trace(go.Scatter(x=df.index, y=df[col], name=col), row=2, col=1)
    # Add reference lines as horizontal lines on each subplot
    main_cc_set = get_setpoint('set_MAIN_CC_WATER', 1.5)
    main_hc_set = get_setpoint('set_MAIN_HC_WATER', 0.0)
    oa_cc_set   = get_setpoint('set_OA_CC_WATER', 0.2)
    oa_hc_set   = get_setpoint('set_OA_HC_WATER', 0.0)
    cw_pump_set = get_setpoint('set_CW_PUMP_FLOW', 3.0)
    hw_pump_set = get_setpoint('set_HW_PUMP_FLOW', 1.0)
    fig7.add_hline(y=main_cc_set, line_dash="dash", line_color=colors[0],
                   annotation_text=f"MC set {main_cc_set:.2f}", row=1, col=1)
    fig7.add_hline(y=main_hc_set, line_dash="dash", line_color=colors[1],
                   annotation_text=f"MH set {main_hc_set:.2f}", row=1, col=1)
    fig7.add_hline(y=oa_cc_set, line_dash="dash", line_color=colors[2],
                   annotation_text=f"OC set {oa_cc_set:.2f}", row=1, col=1)
    fig7.add_hline(y=oa_hc_set, line_dash="dash", line_color=colors[3],
                   annotation_text=f"OH set {oa_hc_set:.2f}", row=1, col=1)
    fig7.add_hline(y=cw_pump_set, line_dash="dash", line_color="cyan",
                   annotation_text=f"CW pump {cw_pump_set:.2f}", row=2, col=1)
    fig7.add_hline(y=hw_pump_set, line_dash="dash", line_color="orange",
                   annotation_text=f"HW pump {hw_pump_set:.2f}", row=2, col=1)
    fig7.update_layout(template=dark_template, hovermode='x unified',
                       title="Coil & Pump Water Flow Rates")
    fig7.update_yaxes(title_text="kg/s", row=1, col=1)
    fig7.update_yaxes(title_text="kg/s", row=2, col=1)
    fig7.show()

plot_dark_mode_results(df)

In [37]:
# @title CSV Save

if res == 0:
    print("Simulation complete. Converting data...")
    df = pd.DataFrame(sim.collected_data)
    sim_start = pd.Timestamp("2026-01-01 00:00:00")
    df['datetime'] = sim_start + pd.to_timedelta(df['day']-1, unit='D') + pd.to_timedelta(df['hour'] + df['minute']/60, unit='h')
    df.set_index('datetime', inplace=True)
    print("DataFrame ready.")

    # Save to CSV
    df.to_csv("simulation_results.csv", index=True)
    print("Data saved to simulation_results.csv")

Simulation complete. Converting data...
DataFrame ready.
Data saved to simulation_results.csv


In [ ]:
# @title
# Get all actuators available in the model
catalog = sim.api_catalog_df()
actuators = catalog['ACTUATORS']   # a DataFrame

# ---- For VAV terminals ----
comp_type = 'AirTerminal:SingleDuct:VAV:Reheat'
vav_act = actuators[actuators['ComponentType'] == comp_type]
print(f"\n=== Valid control types for {comp_type} ===")
print(vav_act[['ControlType', 'ActuatorKey']].drop_duplicates().to_string())

# ---- For reheat coils ----
comp_type = 'Coil:Heating:Water'
coil_act = actuators[actuators['ComponentType'] == comp_type]
# Filter only the zone coils (ignore OA & main coil for now)
zone_coils = coil_act[coil_act['ActuatorKey'].str.contains('ZONE COIL')]
print(f"\n=== Valid control types for {comp_type} (zone reheat coils) ===")
print(zone_coils[['ControlType', 'ActuatorKey']].drop_duplicates().to_string())